# Optuna Tuning for CatBoost — Colab GPU

**Goal:** find the best CatBoost hyperparameters for the Sparkov fraud-detection task
in ~30-50 minutes (vs ~5 hours on local CPU).

**Setup checklist (do these BEFORE running):**
1. **Runtime → Change runtime type → Hardware accelerator → GPU** (T4 is fine; A100 if Pro)
2. Upload `fraudTrain.csv` and `fraudTest.csv` to your Google Drive, e.g. to
   `MyDrive/fraud-detection/data/raw/`
3. Run cell 1 (mount Drive) — paste the auth code when prompted

**What this notebook does:**
1. Mount Drive + load data
2. Replicate the feature engineering from `features.ipynb`
3. Train a baseline CatBoost on GPU (sanity check, ~2 min)
4. Run 50 Optuna trials on GPU (~30-50 min)
5. Re-train with the best params (~2 min)
6. Save the best params as JSON to Drive + auto-download to your machine

**Output:** `catboost_best_params.json` — paste the contents into
`CAT_TUNED` in `features.ipynb` cell 6.


In [1]:
# ── 1. Setup ────────────────────────────────────────────────────────────
# Install/verify packages (catboost, optuna are usually pre-installed on Colab)
!pip install -q catboost optuna 2>&1 | tail -3

import pandas as pd
import numpy as np
import catboost as cb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             fbeta_score, f1_score, precision_score, recall_score)
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.pyplot as plt
import time, gc, warnings, json
warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', 100)
%matplotlib inline


# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "No GPU detected"

DEVICE = 'CPU'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 25.9 MB/s eta 0:00:00
No GPU detected


In [3]:

# Mount Google Drive
from google.colab import drive, files
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ── 2. Load data + time-based split ────────────────────────────────────
# Adjust the path if your Drive folder structure is different.
DATA_DIR = '/content/drive/MyDrive/data/fraud'
TRAIN_PATH = f'{DATA_DIR}/fraudTrain.csv'
TEST_PATH  = f'{DATA_DIR}/fraudTest.csv'

train_full = pd.read_csv(TRAIN_PATH).drop(columns=['Unnamed: 0'])
test_full  = pd.read_csv(TEST_PATH ).drop(columns=['Unnamed: 0'])

train_full['trans_date_trans_time'] = pd.to_datetime(train_full['trans_date_trans_time'])
test_full ['trans_date_trans_time'] = pd.to_datetime(test_full ['trans_date_trans_time'])

print(f"Train: {train_full.shape[0]:>9,} rows | fraud: {train_full.is_fraud.sum():>5,} ({train_full.is_fraud.mean():.4%})")
print(f"Test : {test_full.shape[0]:>9,} rows | fraud: {test_full.is_fraud.sum():>5,} ({test_full.is_fraud.mean():.4%})")

# Time-based split: last 20% of train as val
train_full = train_full.sort_values('trans_date_trans_time').reset_index(drop=True)
cutoff = train_full['trans_date_trans_time'].quantile(0.80)
val_df   = train_full[train_full['trans_date_trans_time'] >= cutoff].reset_index(drop=True)
train_df = train_full[train_full['trans_date_trans_time'] <  cutoff].reset_index(drop=True)

print(f"\nTrain: {len(train_df):>9,} | fraud: {train_df.is_fraud.sum():>4,} ({train_df.is_fraud.mean():.4%})")
print(f"Val  : {len(val_df):>9,} | fraud: {val_df.is_fraud.sum():>4,} ({val_df.is_fraud.mean():.4%})")
print(f"Test : {len(test_full):>9,} | fraud: {test_full.is_fraud.sum():>4,} ({test_full.is_fraud.mean():.4%})")


Train: 1,296,675 rows | fraud: 7,506 (0.5789%)
Test :   555,719 rows | fraud: 2,145 (0.3860%)

Train: 1,037,340 | fraud: 5,968 (0.5753%)
Val  :   259,335 | fraud: 1,538 (0.5931%)
Test :   555,719 | fraud: 2,145 (0.3860%)


In [5]:
# ── 3. Feature engineering (same as features.ipynb) ────────────────────
def build_base_features(df):
    df = df.copy()
    df['hour']     = df['trans_date_trans_time'].dt.hour.astype('int8')
    df['dow']      = df['trans_date_trans_time'].dt.dayofweek.astype('int8')
    df['month']    = df['trans_date_trans_time'].dt.month.astype('int8')
    df['is_night'] = df['hour'].isin([0,1,2,3,4,22,23]).astype('int8')
    df['dob']      = pd.to_datetime(df['dob'])
    df['age']      = ((df['trans_date_trans_time'] - df['dob']).dt.days / 365.25).astype('float32')
    df['amt_log']  = np.log1p(df['amt']).astype('float32')
    df['amt_is_round'] = (df['amt'] == df['amt'].round(0)).astype('int8')
    R = 6371.0
    lat1, lon1 = np.radians(df['lat']),     np.radians(df['long'])
    lat2, lon2 = np.radians(df['merch_lat']), np.radians(df['merch_long'])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    df['distance_km'] = (2 * R * np.arcsin(np.sqrt(a))).astype('float32')
    return df

train_df = build_base_features(train_df)
val_df   = build_base_features(val_df)
test_df  = build_base_features(test_full)

combined = pd.concat([train_df, val_df, test_df], ignore_index=True)
for col in ['cc_num', 'merchant', 'category', 'city', 'state', 'job', 'zip']:
    counts = combined[col].value_counts().to_dict()
    for d in (train_df, val_df, test_df):
        d[f'{col}_FE'] = d[col].map(counts).astype('float32')
del combined; gc.collect()

GLOBAL = train_df['is_fraud'].mean()
SMOOTHING = 50.0
for col in ['merchant', 'category', 'city', 'state', 'job']:
    stats = train_df.groupby(col)['is_fraud'].agg(['mean', 'count'])
    smoothed = (stats['mean'] * stats['count'] + GLOBAL * SMOOTHING) / (stats['count'] + SMOOTHING)
    smoothed = smoothed.to_dict()
    for d in (train_df, val_df, test_df):
        d[f'{col}_te'] = d[col].map(smoothed).fillna(GLOBAL).astype('float32')

for col in ['merchant', 'category', 'cc_num']:
    stats = train_df.groupby(col)['amt'].agg(['mean', 'std'])
    for stat in ['mean', 'std']:
        colname = f'amt_per_{col}_{stat}'
        mapping = stats[stat].to_dict()
        fallback = train_df['amt'].mean() if stat == 'mean' else train_df['amt'].std()
        for d in (train_df, val_df, test_df):
            d[colname] = d[col].map(mapping).fillna(fallback).astype('float32')

def add_velocity(df, window_hours_list):
    df = df.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)
    df['ts'] = df['trans_date_trans_time'].astype('int64') // 10**9
    for h in window_hours_list:
        window_sec = h * 3600
        df[f'txn_last_{h}h'] = (
            df.groupby('cc_num')['ts']
              .transform(lambda s: s.searchsorted(s.values - h*3600, side='right') - 1)
              .astype('float32')
        )
        amt_sums = np.zeros(len(df), dtype='float32')
        for _, g in df.groupby('cc_num', sort=False):
            ts = g['ts'].values; amt = g['amt'].values; idx = g.index.values; n = len(g)
            cum_amt = np.concatenate([[0.0], np.cumsum(amt, dtype='float64')])
            for i in range(n):
                j = np.searchsorted(ts[:i+1], ts[i] - window_sec, side='left')
                amt_sums[idx[i]] = cum_amt[i] - cum_amt[j]
        df[f'amt_sum_last_{h}h'] = amt_sums
    return df.drop(columns=['ts'])

train_df = add_velocity(train_df, [1, 24, 168])
val_df   = add_velocity(val_df,   [1, 24, 168])
test_df  = add_velocity(test_df,  [1, 24, 168])

DROP = ['trans_date_trans_time', 'first', 'last', 'street', 'dob',
        'trans_num', 'unix_time', 'lat', 'long', 'merch_lat', 'merch_long',
        'cc_num', 'merchant', 'category', 'city', 'state', 'job', 'gender', 'zip',
        'is_fraud']
FEATURES = [c for c in train_df.columns if c not in DROP]
print(f"Total features: {len(FEATURES)}")

X_train, y_train = train_df[FEATURES].values, train_df['is_fraud'].values
X_val,   y_val   = val_df  [FEATURES].values, val_df  ['is_fraud'].values
X_test,  y_test  = test_df [FEATURES].values, test_df ['is_fraud'].values
print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")


Total features: 34
X_train: (1037340, 34)  X_val: (259335, 34)  X_test: (555719, 34)


In [6]:
# ── 4. Baseline CatBoost on GPU (sanity check) ─────────────────────────
# Quick run to confirm GPU is working and we have a baseline PR-AUC.
print(f"Running on: {DEVICE}")
t0 = time.time()
m_base = cb.CatBoostClassifier(
    iterations=500, depth=8, learning_rate=0.05,
    eval_metric='PRAUC', random_seed=42, verbose=0,
    early_stopping_rounds=50, task_type=DEVICE,
)
m_base.fit(X_train, y_train, eval_set=(X_val, y_val))
t_base = time.time() - t0

val_pr  = average_precision_score(y_val,  m_base.predict_proba(X_val) [:, 1])
test_pr = average_precision_score(y_test, m_base.predict_proba(X_test)[:, 1])
test_roc = roc_auc_score(y_test, m_base.predict_proba(X_test)[:, 1])

print(f"\nBaseline CatBoost ({DEVICE}) | fit {t_base:.1f}s")
print(f"  val  PR-AUC : {val_pr:.4f}")
print(f"  test PR-AUC : {test_pr:.4f}")
print(f"  test ROC-AUC: {test_roc:.4f}")
print(f"\nExpected per-trial time on {DEVICE}: ", end='')
if DEVICE == 'GPU':
    print("30-60 sec")
else:
    print("5-7 min (CPU is slow; consider fewer trials)")


Running on: CPU

Baseline CatBoost (CPU) | fit 293.2s
  val  PR-AUC : 0.9431
  test PR-AUC : 0.8881
  test ROC-AUC: 0.9952

Expected per-trial time on CPU: 5-7 min (CPU is slow; consider fewer trials)


In [7]:
# ── 5. Optuna search (50 trials, GPU) ──────────────────────────────────
# Search space: focused on the 5 most impactful CatBoost params.
# Other params (loss_function, eval_metric, etc.) are fixed.
N_TRIALS = 50 if DEVICE == 'GPU' else 30  # fewer trials on CPU
print(f"Running {N_TRIALS} Optuna trials on {DEVICE}")
print(f"Estimated total time: ", end='')
if DEVICE == 'GPU':
    print(f"{N_TRIALS * 0.75:.0f}-{N_TRIALS * 1.5:.0f} min")
else:
    print(f"{N_TRIALS * 5:.0f}-{N_TRIALS * 7:.0f} min")
print()

def objective(trial):
    params = {
        'iterations':         trial.suggest_int('iterations', 300, 1500),
        'depth':              trial.suggest_int('depth', 4, 10),
        'learning_rate':      trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'l2_leaf_reg':        trial.suggest_float('l2_leaf_reg', 0.5, 20.0, log=True),
        'random_strength':    trial.suggest_float('random_strength', 0.0, 5.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 2.0),
        'border_count':       trial.suggest_int('border_count', 32, 254),
        'eval_metric':        'PRAUC',
        'random_seed':        42,
        'verbose':            0,
        'early_stopping_rounds': 30,  # aggressive early stop to save time
        'task_type':          DEVICE,
    }
    m = cb.CatBoostClassifier(**params)
    m.fit(X_train, y_train, eval_set=(X_val, y_val))
    return average_precision_score(y_val, m.predict_proba(X_val)[:, 1])

t0 = time.time()
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=10),
)
# Add a callback to print progress
def print_best(study, trial):
    if (trial.number + 1) % 5 == 0 or trial.number == 0:
        print(f"  Trial {trial.number+1:3d}/{N_TRIALS} | best so far: {study.best_value:.4f} | "
              f"this trial: {trial.value:.4f} | elapsed: {time.time()-t0:.0f}s")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False, callbacks=[print_best])

t_optuna = time.time() - t0
print(f"\nOptuna complete in {t_optuna/60:.1f} min")
print(f"Best val PR-AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")


Running 30 Optuna trials on CPU
Estimated total time: 150-210 min

  Trial   1/30 | best so far: 0.8975 | this trial: 0.8975 | elapsed: 125s
  Trial   5/30 | best so far: 0.9077 | this trial: 0.7951 | elapsed: 296s
  Trial  10/30 | best so far: 0.9313 | this trial: 0.8032 | elapsed: 1057s
  Trial  15/30 | best so far: 0.9432 | this trial: 0.9392 | elapsed: 1817s
  Trial  20/30 | best so far: 0.9432 | this trial: 0.8610 | elapsed: 2378s
  Trial  25/30 | best so far: 0.9432 | this trial: 0.9265 | elapsed: 2846s
  Trial  30/30 | best so far: 0.9432 | this trial: 0.9328 | elapsed: 3568s

Optuna complete in 59.5 min
Best val PR-AUC: 0.9432
Best params: {'iterations': 307, 'depth': 10, 'learning_rate': 0.14757878491291962, 'l2_leaf_reg': 19.613363614465985, 'random_strength': 1.2157708813939343, 'bagging_temperature': 1.1713021441934048, 'border_count': 192}


In [8]:
# ── 6. Re-train final model with best params (full data, GPU) ──────────
# Train one more time with the best params, slightly more iterations.
best_params = dict(study.best_params)
best_params.update({
    'iterations':    best_params['iterations'],  # keep what Optuna found
    'eval_metric':   'PRAUC',
    'random_seed':   42,
    'verbose':       0,
    'task_type':     DEVICE,
})

t0 = time.time()
m_final = cb.CatBoostClassifier(**best_params)
m_final.fit(X_train, y_train, eval_set=(X_val, y_val))
t_final = time.time() - t0

val_pr  = average_precision_score(y_val,  m_final.predict_proba(X_val) [:, 1])
test_pr = average_precision_score(y_test, m_final.predict_proba(X_test)[:, 1])
test_roc = roc_auc_score(y_test, m_final.predict_proba(X_test)[:, 1])

print(f"Final CatBoost (Optuna-tuned, {DEVICE}) | fit {t_final:.1f}s")
print(f"  val  PR-AUC : {val_pr:.4f}")
print(f"  test PR-AUC : {test_pr:.4f}")
print(f"  test ROC-AUC: {test_roc:.4f}")
print(f"\nDelta vs baseline: test PR-AUC {test_pr - 0.8881:+.4f} (assuming baseline=0.8881)")


Final CatBoost (Optuna-tuned, CPU) | fit 350.3s
  val  PR-AUC : 0.9435
  test PR-AUC : 0.8856
  test ROC-AUC: 0.9957

Delta vs baseline: test PR-AUC -0.0025 (assuming baseline=0.8881)


In [9]:
# ── 7. Save best params + download ─────────────────────────────────────
# Build the CAT_TUNED dict that features.ipynb cell 6 expects
cat_tuned = {
    'iterations':    study.best_params['iterations'],
    'depth':         study.best_params['depth'],
    'learning_rate': study.best_params['learning_rate'],
    'l2_leaf_reg':   study.best_params['l2_leaf_reg'],
    'random_strength':   study.best_params['random_strength'],
    'bagging_temperature': study.best_params['bagging_temperature'],
    'border_count':  study.best_params['border_count'],
    'eval_metric':   'PRAUC',
    'random_seed':   42,
    'verbose':       0,
    'early_stopping_rounds': 50,
    'task_type':     'CPU',  # local features.ipynb uses CPU
}

# Save to Drive
save_path = '/content/drive/MyDrive/data/catboost_best_params.json'
import os
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, 'w') as f:
    json.dump(cat_tuned, f, indent=2)
print(f"Saved to Drive: {save_path}")

# Also auto-download to local
print("\nDownloading to your local machine...")
files.download(save_path)

# Print the dict (for manual copy-paste if download fails)
print("\n" + "=" * 70)
print("COPY THIS INTO features.ipynb CELL 6 (replaces CAT_TUNED):")
print("=" * 70)
print("CAT_TUNED = " + repr(cat_tuned))


Saved to Drive: /content/drive/MyDrive/data/catboost_best_params.json



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


COPY THIS INTO features.ipynb CELL 6 (replaces CAT_TUNED):
CAT_TUNED = {'iterations': 307, 'depth': 10, 'learning_rate': 0.14757878491291962, 'l2_leaf_reg': 19.613363614465985, 'random_strength': 1.2157708813939343, 'bagging_temperature': 1.1713021441934048, 'border_count': 192, 'eval_metric': 'PRAUC', 'random_seed': 42, 'verbose': 0, 'early_stopping_rounds': 50, 'task_type': 'CPU'}


## Summary

**Output:** `catboost_best_params.json` containing the Optuna-tuned CatBoost params.

**Next steps (on your local machine):**
1. Open `notebooks/features.ipynb` in your local environment
2. Replace the `CAT_TUNED` dict in cell 6 with the values from the JSON
3. Re-execute the full notebook — the "tuned" cell 7 will use the real Optuna params
4. Feature selection methods (cells 9-14) will use the better model

**Expected improvement** (vs the placeholder `CAT_TUNED`):
- The placeholder scored test PR-AUC = 0.8820
- Your local baseline (cell 5) scored test PR-AUC = 0.8881
- Real Optuna should land in the 0.88-0.91 range, with 0.89+ being a good result

**Troubleshooting:**
- If GPU is unavailable: cell 1 auto-falls back to CPU with 20 trials (~2-3 hr)
- If Colab disconnects: the study isn't saved; re-run cell 5 to restart
- If you get a CUDA OOM: try `!nvidia-smi` to check VRAM; the dataset is small
  so it should fit on any Colab GPU

**Alternative: skip Optuna and use baseline params**
The local baseline (0.8881) is already strong. If Optuna doesn't help
meaningfully, just use the baseline params in `CAT_TUNED` and move on to
feature selection.
